In [1]:
import pandas as pd
import re
import ast
from collections import Counter

import spacy

In [2]:
nlp = spacy.load("en_core_web_sm")

print("spaCy Loaded Successfully")

spaCy Loaded Successfully


In [3]:
jobs = pd.read_csv("../data/jobs_with_skills.csv")

jobs.head()

,job_id,company_name,title,description,location,formatted_work_type,formatted_experience_level,min_salary,max_salary,med_salary,currency,remote_allowed,skills_desc,clean_description,clean_title,extracted_skills
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,"Princeton, NJ",Full-time,Not Available,17.0,20.0,0.0,USD,0.0,Requirements: \n\nWe are seeking a College or ...,job descriptiona leading real estate firm in n...,marketing coordinator,[]
1,1829192,Not Available,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...","Fort Collins, CO",Full-time,Not Available,30.0,50.0,0.0,USD,0.0,Not Available,at aspen therapy and wellness we are committed...,mental health therapist counselor,['excel']
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,"Cincinnati, OH",Full-time,Not Available,45000.0,65000.0,0.0,USD,0.0,We are currently accepting resumes for FOH - A...,the national exemplar is accepting application...,assitant restaurant manager,[]
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,"New Hyde Park, NY",Full-time,Not Available,140000.0,175000.0,0.0,USD,0.0,This position requires a baseline understandin...,senior associate attorney elder law trusts and...,senior elder law trusts and estates associate ...,['excel']
4,35982263,Not Available,Service Technician,Looking for HVAC service tech with experience ...,"Burlington, IA",Full-time,Not Available,60000.0,80000.0,0.0,USD,0.0,Not Available,looking for hvac service tech with experience ...,service technician,[]


In [4]:
SKILLS = [

    # Programming
    "python","java","c++","c#","javascript","typescript",
    "r","scala","go","ruby","php","matlab",

    # Web
    "html","css","react","angular","vue","node.js",
    "django","flask","spring","spring boot",

    # Databases
    "sql","mysql","postgresql","mongodb","oracle",
    "sqlite","redis","snowflake",

    # Data Science
    "pandas","numpy","scikit-learn","tensorflow",
    "pytorch","machine learning","deep learning",
    "data science","data analysis","statistics",

    # Visualization
    "power bi","tableau","matplotlib","seaborn",
    "plotly","excel",

    # Big Data
    "spark","hadoop","databricks","airflow",

    # Cloud
    "aws","azure","gcp","google cloud",

    # DevOps
    "docker","kubernetes","terraform",
    "jenkins","github actions","git",

    # Operating Systems
    "linux","unix",

    # AI
    "nlp","computer vision","generative ai",
    "llm","openai","langchain","rag",

    # Business
    "project management","agile","scrum",
    "communication","leadership",

    # Finance
    "financial analysis","accounting",
    "budgeting","forecasting",

    # Marketing
    "seo","sem","google analytics",
    "content marketing","social media marketing"
]

In [5]:
skill_map = {
    "js":"javascript",
    "javascript":"javascript",

    "py":"python",
    "python":"python",

    "postgres":"postgresql",
    "postgresql":"postgresql",

    "google cloud platform":"gcp",
    "google cloud":"gcp",

    "powerbi":"power bi",

    "machine-learning":"machine learning",

    "artificial intelligence":"ai"
}

In [6]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [7]:
import re

# Create regex once
skill_regex = re.compile(
    r"\b(" + "|".join(map(re.escape, SKILLS)) + r")\b",
    re.IGNORECASE
)

def extract_skills(text):

    text = str(text).lower()

    matches = skill_regex.findall(text)

    normalized = [
        skill_map.get(skill, skill)
        for skill in matches
    ]

    return sorted(set(normalized))


from tqdm.auto import tqdm

tqdm.pandas()

jobs["extracted_skills"] = jobs[
    "clean_description"
].progress_apply(extract_skills)

C:\Users\khush\OneDrive\Desktop\AI-Powered-Job-Market-Intelligence-Platform\AI-Powered-Job-Market-Intelligence-Platform\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 123849/123849 [07:54<00:00, 260.88it/s]


In [8]:
jobs["skill_count"] = jobs[
    "extracted_skills"
].apply(len)

jobs["skill_count"].describe()

count    123849.000000
mean          1.716259
std           1.973719
min           0.000000
25%           0.000000
50%           1.000000
75%           2.000000
max          31.000000
Name: skill_count, dtype: float64

In [9]:
jobs[
    ["title","extracted_skills"]
].sample(20)

,title,extracted_skills
48920,Middle Office Manager - Top Multi-Strat Hedge ...,"[go, leadership, project management, python, sql]"
108410,Job opportunities- Charlotte Douglas Internati...,[]
45860,"Senior Associate, Information Security",[]
19083,DAU Manager - NA Growth Operations,"[agile, communication, leadership]"
106028,Business Development Manager (MA),"[communication, leadership, project management]"
93857,Footwear Sales Outfitter,[]
74648,"Technical Lead, Order Fulfillment","[aws, azure, gcp, java]"
2644,Controls Engineer,[communication]
60822,Production Worker,[]
45949,Certified Nursing Assistant PRN on the Surgica...,[]


In [10]:
jobs["skill_count"].mean()

np.float64(1.7162593157796995)

In [11]:
jobs["skill_count"].max()

np.int64(31)

In [12]:
jobs[["title","extracted_skills"]].sample(5)

,title,extracted_skills
115920,Senior Oracle Database Administrator,[oracle]
30750,ServiceNow CMDB Engineer,"[communication, excel]"
63770,Plant Operator,[]
8033,"Senior Software Engineer, Project Starline",[leadership]
102562,Field Inspector,[]


In [13]:
jobs.shape

(123849, 17)

In [14]:
zero_skill_jobs = (
    jobs["skill_count"] == 0
).sum()

zero_skill_percentage = (
    zero_skill_jobs / len(jobs) * 100
)

zero_skill_percentage

np.float64(26.76485074566609)

In [15]:
from collections import Counter

skill_counts = Counter(
    skill
    for skills in jobs["extracted_skills"]
    for skill in skills
)

top_skills = pd.DataFrame(
    skill_counts.most_common(20),
    columns=["Skill", "Demand"]
)

top_skills

,Skill,Demand
0,communication,59998
1,leadership,29355
2,excel,18107
3,project management,10249
4,go,8830
5,accounting,8802
6,agile,5969
7,r,5229
8,sql,5183
9,python,4648


In [16]:
top_skills["Demand_Percentage"] = (
    top_skills["Demand"] / len(jobs) * 100
)

top_skills

,Skill,Demand,Demand_Percentage
0,communication,59998,48.444477
1,leadership,29355,23.702250
2,excel,18107,14.620223
3,project management,10249,8.275400
4,go,8830,7.129650
5,accounting,8802,7.107042
6,agile,5969,4.819579
7,r,5229,4.222077
8,sql,5183,4.184935
9,python,4648,3.752957


In [17]:
soft_skills = [
    "communication",
    "leadership",
    "project management",
    "agile",
    "scrum"
]

In [18]:
technical_skill_counts = {
    skill: count
    for skill, count in skill_counts.items()
    if skill not in soft_skills
}

In [19]:
top_technical_skills = pd.DataFrame(
    sorted(
        technical_skill_counts.items(),
        key=lambda x: x[1],
        reverse=True
    )[:20],
    columns=["Skill", "Demand"]
)

top_technical_skills

,Skill,Demand
0,excel,18107
1,go,8830
2,accounting,8802
3,r,5229
4,sql,5183
5,python,4648
6,data analysis,3273
7,aws,3161
8,forecasting,2976
9,azure,2918


In [20]:
top_technical_skills["Demand_Percentage"] = (
    top_technical_skills["Demand"] / len(jobs) * 100
)

top_technical_skills

,Skill,Demand,Demand_Percentage
0,excel,18107,14.620223
1,go,8830,7.129650
2,accounting,8802,7.107042
3,r,5229,4.222077
4,sql,5183,4.184935
5,python,4648,3.752957
6,data analysis,3273,2.642734
7,aws,3161,2.552302
8,forecasting,2976,2.402926
9,azure,2918,2.356095


In [21]:
soft_skill_counts = {
    skill: count
    for skill, count in skill_counts.items()
    if skill in soft_skills
}

top_soft_skills = pd.DataFrame(
    sorted(
        soft_skill_counts.items(),
        key=lambda x: x[1],
        reverse=True
    ),
    columns=["Skill", "Demand"]
)

top_soft_skills

,Skill,Demand
0,communication,59998
1,leadership,29355
2,project management,10249
3,agile,5969
4,scrum,1556


In [22]:
top_soft_skills["Demand_Percentage"] = (
    top_soft_skills["Demand"] / len(jobs) * 100
)

top_soft_skills

,Skill,Demand,Demand_Percentage
0,communication,59998,48.444477
1,leadership,29355,23.702250
2,project management,10249,8.275400
3,agile,5969,4.819579
4,scrum,1556,1.256369


In [23]:
jobs.to_csv(
    "../data/jobs_with_skills_v2.csv",
    index=False
)

In [24]:
sample_go = jobs[jobs["extracted_skills"].apply(lambda x: "go" in x)]["clean_description"].sample(5, random_state=1)
for text in sample_go:
    print(text[:200], "\n---")

are you an experienced family law paralegal looking for a new opportunity in the world of family law the litigation family law paralegal position is the perfect fit if you re ready to take on a new ch 
---
we re a little different our mission is clear we bring to life a healing ministry through our compassionate care and exceptional service at mercy we believe in careers that match the unique gifts of u 
---
we re a little different our mission is clear we bring to life a healing ministry through our compassionate care and exceptional service at mercy we believe in careers that match the unique gifts of u 
---
elastic is a free and open search company that powers enterprise search observability and security solutions built on one technology stack that can be deployed anywhere from finding documents to monit 
---
business specialist our client is adding an energetic person looking to grow their career this position will give you the fundamentals for success with one of omaha s best comp

In [27]:
# Find where "go" is defined in your SKILLS list, e.g.:
SKILLS = [
    
    "go", 
    
]

# Change it to:
SKILLS = [
    
    "golang",   # much safer — almost nobody writes "golang" unless they mean the language
    
]

In [28]:

jobs.to_csv(
    "../data/jobs_with_skills_v2.csv",
    index=False
)